In [ ]:
!pip install --upgrade langchain langchain-core transformers langsmith

In [ ]:
import os
from getpass import getpass

os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith Key: ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [ ]:
import os
os.makedirs("data", exist_ok=True)

# Strong Resume
with open("data/strong_resume.txt", "w") as f:
    f.write("""Name: Rahul Sharma

Skills: Python, Machine Learning, Deep Learning, NLP, SQL, TensorFlow
Tools: Pandas, NumPy, Scikit-learn, Power BI
Experience: 3 years as Data Scientist
""")

# Average Resume
with open("data/average_resume.txt", "w") as f:
    f.write("""Name: Priya Singh

Skills: Python, SQL, Data Analysis
Tools: Excel, Pandas
Experience: 1 year as Data Analyst
""")

# Weak Resume
with open("data/weak_resume.txt", "w") as f:
    f.write("""Name: Amit Kumar

Skills: MS Word, Communication
Tools: None
Experience: Fresher
""")

# Job Description
with open("data/job_description.txt", "w") as f:
    f.write("""Looking for Data Scientist with:
- Python, Machine Learning, NLP
- TensorFlow or PyTorch
- SQL knowledge
- 2+ years experience
""")

In [ ]:
!pip install langchain-community

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline("text-generation", model="google/flan-t5-base")

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.prompts import PromptTemplate

# Extract
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract skills, tools, and experience from the resume.
Do NOT assume anything.

Resume:
{resume}
"""
)

# Match
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_desc"],
    template="""
Compare resume with job description.

Return:
- matched_skills
- missing_skills
- experience_match

Resume:
{resume_data}

Job:
{job_desc}
"""
)

# Score
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give a score from 0 to 100 based on matching.

Matching:
{match_data}
"""
)

# Explain
explain_prompt = PromptTemplate(
    input_variables=["match_data", "score"],
    template="""
Explain the score clearly.

Score: {score}

Matching:
{match_data}
"""
)

In [ ]:
extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm
explain_chain = explain_prompt | llm

In [ ]:
files = [
    "data/strong_resume.txt",
    "data/average_resume.txt",
    "data/weak_resume.txt"
]

job_desc = open("data/job_description.txt").read()

for file in files:
    print("\n==============================")
    print("Processing:", file)
    print("==============================")

    resume = open(file).read()

    # Step 1: Extract
    resume_data = extract_chain.invoke({"resume": resume})
    print("\n[Extracted Data]")
    print(resume_data)

    # Step 2: Match
    match_data = match_chain.invoke({
        "resume_data": resume_data,
        "job_desc": job_desc
    })
    print("\n[Matching Data]")
    print(match_data)

    # Step 3: Score
    score = score_chain.invoke({
        "match_data": match_data
    })
    print("\n[Score]")
    print(score)

    # Step 4: Explain
    explanation = explain_chain.invoke({
        "match_data": match_data,
        "score": score
    })
    print("\n[Explanation]")
    print(explanation)

    print("\n====================================\n")